# HEAD-QA v2 (alesi12/head_qa_v2) — Data Inspection

Spanish and Italian healthcare professional MCQ.
Only a single 'train' split (12.8k rows per language).
We apply our own 85/15 train/val carve-out.

Key concerns:
- Image questions (~14% of medicine) — filter these
- 2013-2014 have 5 options, 2015+ have 4 — both acceptable
- ra is 1-indexed int matching aid — map to letter by position
- Italian config is machine-translated (Llama-3.1-8B) — spot-check quality

In [31]:
from datasets import load_dataset
import pandas as pd
import re
import json
from pathlib import Path
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

TARGET_CONFIGS = {'es': 'spanish', 'it': 'italian'}
LETTERS = 'ABCDE'

In [11]:
# Load both configs
datasets = {}
for config in TARGET_CONFIGS:
    ds = load_dataset('alesi12/head_qa_v2', config)
    print(f'{config}: {ds}')
    datasets[config] = ds

es: DatasetDict({
    train: Dataset({
        features: ['qid', 'qtext', 'ra', 'answers', 'year', 'category', 'name', 'image'],
        num_rows: 12751
    })
})


it/train-00000-of-00001.parquet:   0%|          | 0.00/91.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/12751 [00:00<?, ? examples/s]

it: DatasetDict({
    train: Dataset({
        features: ['qid', 'qtext', 'ra', 'answers', 'year', 'category', 'name', 'image'],
        num_rows: 12751
    })
})


In [32]:
# Inspect raw structure
ex = datasets['es']['train'][0]
print('Keys:', list(ex.keys()))
print()
for k, v in ex.items():
    if k != 'image':
        print(f'{k}: {repr(v)[:200]}')
print(f'image: {type(ex["image"])} (is None: {ex["image"] is None})')

Keys: ['qid', 'qtext', 'ra', 'answers', 'year', 'category', 'name', 'image']

qid: 1
qtext: 'Los potenciales postsinápticos excitadores:'
ra: 3
answers: [{'aid': 1, 'atext': 'Son de tipo todo o nada.'}, {'aid': 2, 'atext': 'Son hiperpolarizantes.'}, {'aid': 3, 'atext': 'Se pueden sumar.'}, {'aid': 4, 'atext': 'Se propagan a largas distancias.'}, {'aid
year: 2013
category: 'biology'
name: 'Cuaderno_2013_1_B'
image: <class 'NoneType'> (is None: True)


In [33]:
# flatten all configs into one dataframe
rows = []
for config, lang in TARGET_CONFIGS.items():
    for ex in datasets[config]['train']:
        answers = ex['answers']
        ra = int(ex['ra'])
        has_image = ex['image'] is not None

        # Map ra to letter by position (ra matches aid, aids are 1-indexed)
        ra_idx = next((i for i, a in enumerate(answers) if a['aid'] == ra), None)
        ra_letter = LETTERS[ra_idx] if ra_idx is not None else None

        rows.append({
            'config':    config,
            'language':  lang,
            'qid':       ex['qid'],
            'name':      ex['name'],
            'year':      ex['year'],
            'category':  ex['category'],
            'qtext':     ex['qtext'],
            'n_answers': len(answers),
            'ra':        ra,
            'ra_letter': ra_letter,
            'has_image': has_image,
            'aids':      [a['aid'] for a in answers],
            'atexts':    [a['atext'] for a in answers],
        })

df = pd.DataFrame(rows)
print(df.shape)
df.head(2)

(25502, 13)


,config,language,qid,name,year,category,qtext,n_answers,ra,ra_letter,has_image,aids,atexts
0,es,spanish,1,Cuaderno_2013_1_B,2013,biology,Los potenciales postsinápticos excitadores:,5,3,C,False,"[1, 2, 3, 4, 5]","[Son de tipo todo o nada., Son hiperpolarizantes., Se pueden sumar., Se propagan a largas distancias., Presentan un periodo refractario.]"
1,es,spanish,2,Cuaderno_2013_1_B,2013,biology,Placa motora es la unión entre la neurona motora y el:,5,2,B,False,"[1, 2, 3, 4, 5]","[Músculo liso., Músculo esquelético., Músculo cardiaco., Huso muscular., Tendón.]"


In [ ]:
# basic counts
print(df.groupby(['config', 'language']).size())

config  language
es      spanish     12751
it      italian     12751
dtype: int64


In [21]:
# Category distribution
print(df.groupby(['language', 'category']).size().to_string())

language  category    
italian   biology         2119
          chemistry       2140
          medicine        2156
          nursing         2069
          pharmacology    2146
          psychology      2121
spanish   biology         2119
          chemistry       2140
          medicine        2156
          nursing         2069
          pharmacology    2146
          psychology      2121


In [22]:
# Answer count distribution (4 vs 5 options)
print(df.groupby(['language', 'n_answers']).size().to_string())
print()
print(df.groupby(['year', 'n_answers']).size().to_string())


language  n_answers
italian   4            10094
          5             2657
spanish   4            10094
          5             2657

year  n_answers
2013  5            2636
2014  5            2678
2015  4            2732
2016  4            2744
2017  4            2740
2018  4            2728
2019  4            2156
2020  4            2148
2021  4            2460
2022  4            2480


In [24]:
# Image question analysis
print('Image questions by language + category:')
print(df.groupby(['language', 'category', 'has_image']).size().to_string())

Image questions by language + category:
language  category      has_image
italian   biology       False        2119
          chemistry     False        2140
          medicine      False        1858
                        True          298
          nursing       False        2033
                        True           36
          pharmacology  False        2146
          psychology    False        2121
spanish   biology       False        2119
          chemistry     False        2140
          medicine      False        1858
                        True          298
          nursing       False        2033
                        True           36
          pharmacology  False        2146
          psychology    False        2121


In [25]:
# Check if image questions explicitly reference an image in the text
image_ref = re.compile(
    r'\b(imagen|figura|radiograf|rx|\bfoto\b|\bfig\.?\b|immagine|figura|'
    r'image|figure|radiograph)',
    re.IGNORECASE
)
df['image_ref_in_text'] = df['qtext'].apply(lambda x: bool(image_ref.search(str(x))))

print(df.groupby(['has_image', 'image_ref_in_text']).size())
print()
print(f'Has image + mentions image (must filter): {len(df[df["has_image"] & df["image_ref_in_text"]])}')
print(f'Has image + no image ref (maybe answerable): {len(df[df["has_image"] & ~df["image_ref_in_text"]])}')

has_image  image_ref_in_text
False      False                24596
           True                   238
True       True                   668
dtype: int64

Has image + mentions image (must filter): 668
Has image + no image ref (maybe answerable): 0


In [34]:
# Check ra mapping worked
bad_ra = df[df['ra_letter'].isna()]
print(f'Rows where ra does not match any aid: {len(bad_ra)}')
if len(bad_ra):
    display(bad_ra[['language', 'qid', 'qtext', 'ra', 'aids']].head(10))
# Encoding corruption
corruption = re.compile(r'[\ufffd\x00-\x08\x0b\x0c\x0e-\x1f]')
def has_corruption(row):
    return any(corruption.search(str(f)) for f in [row['qtext']] + list(row['atexts']))

df['has_corruption'] = df.apply(has_corruption, axis=1)
print(f'Encoding corruption by language:')
print(df.groupby(['language', 'has_corruption']).size())

Rows where ra does not match any aid: 56


,language,qid,qtext,ra,aids
9433,spanish,48,"En el glomérulo renal, la membrana basal implicada en la filtración está constituida por células:",0,"[1, 2, 3, 4]"
9439,spanish,56,La hipermutación somática es un proceso que afecta a:,0,"[1, 2, 3, 4]"
9466,spanish,84,El operón lac es un sistema inducible que está bajo control:,0,"[1, 2, 3, 4]"
9467,spanish,85,En relación con los geles de agarosa en la separación de fragmentos de ADN:,0,"[1, 2, 3, 4]"
9500,spanish,121,El precursor común en la síntesis de los esteroides sexuales (andrógenos y estrógenos) es:,0,"[1, 2, 3, 4]"
9504,spanish,126,La inhibina B:,0,"[1, 2, 3, 4]"
9575,spanish,13,Señale la opción correcta con relación al seguimiento farmacoterapéutico y la reevaluación del mismo que se debe realizar a un paciente anticoagulado con acenocumarol:,0,"[1, 2, 3, 4]"
9635,spanish,74,La definición operacional de Sarcopenia tras la revisión del EWGSOP2 de 2018 establece que:,0,"[1, 2, 3, 4]"
9646,spanish,86,¿Qué indicaciones ofrecerá a un paciente intervenido de una artrodesis de rodilla izquierda para caminar con 4 puntos de apoyo utilizando muletas?:,0,"[1, 2, 3, 4]"
9687,spanish,129,Indique cuál de las siguientes NO es una complicación derivada de la aplicación de un taponamiento nasal:,0,"[1, 2, 3, 4]"


Encoding corruption by language:
language  has_corruption
italian   False             12751
spanish   False             12751
dtype: int64


In [35]:
# Italian translation quality spot-check
# Compare same question in ES vs IT
print('=== Spanish vs Italian translation spot-check ===')
es_sample = df[df['language'] == 'spanish'].head(3)
it_sample = df[df['language'] == 'italian'].head(3)
for (_, es), (_, it) in zip(es_sample.iterrows(), it_sample.iterrows()):
    print(f'ES: {es["qtext"][:120]}')
    print(f'IT: {it["qtext"][:120]}')
    print()

=== Spanish vs Italian translation spot-check ===
ES: Los potenciales postsinápticos excitadores:
IT: I potenziali postsinaptici eccitatori:

ES: Placa motora es la unión entre la neurona motora y el:
IT: La placa motrice è l'unione tra la cellula motrice e il:

ES: NO generan potenciales de acción:
IT: NON generano potenziali d'azione:



In [16]:
# Spot-check formatted examples per category
for lang in ['spanish', 'italian']:
    for cat in df[df['language'] == lang]['category'].unique():
        row = df[(df['language'] == lang) & (df['category'] == cat) & ~df['has_image']].iloc[0]
        print(f'=== {lang} / {cat} ({row["year"]}) ===')
        print(f'Q: {row["qtext"]}')
        for i, (aid, atext) in enumerate(zip(row['aids'], row['atexts'])):
            marker = ' ← correct' if aid == row['ra'] else ''
            print(f'  {LETTERS[i]}) {atext}{marker}')
        print()

=== spanish / biology (2013) ===
Q: Los potenciales postsinápticos excitadores:
  A) Son de tipo todo o nada.
  B) Son hiperpolarizantes.
  C) Se pueden sumar. ← correct
  D) Se propagan a largas distancias.
  E) Presentan un periodo refractario.

=== spanish / nursing (2013) ===
Q: ¿Cuáles son los tres tipos de células óseas?
  A) Trabéculas, mitocondrias y osteocitos.
  B) Osteoclastos, trabéculas y osteolitos.
  C) Osteocitos, osteoblastos y mitocondrias.
  D) Trabéculas, mitocondrias y osteocitos.
  E) Osteoblastos, osteoclastos y osteocitos. ← correct

=== spanish / pharmacology (2013) ===
Q: Desde el punto de vista del metabolismo de xenobióticos, ¿a qué se deben la hepatotoxicidad y la acción cancerígena de las aflatoxinas?
  A) Su hidrólisis libera aminas aromáticas.
  B) Su metabolismo de la fase II genera un catión nitrenio, que es citotóxico por reaccionar con el ADN.
  C) Su metabolismo hepático origina un epóxido, que forma un enlace covalente con la guanina del ADN. ← cor

In [36]:
# Cleaning: filter image questions, bad ra mapping, corruption
clean = df[
    ~df['has_image'] &
    df['ra_letter'].notna() &
    ~df['has_corruption'] &
    (df['qtext'].str.strip() != '')
].copy()

rejected = df[~df.index.isin(clean.index)].copy()

print(f'Total:    {len(df)}')
print(f'Clean:    {len(clean)}')
print(f'Rejected: {len(rejected)}')
print()
print(clean.groupby(['language', 'n_answers']).size())

Total:    25502
Clean:    24778
Rejected: 724

language  n_answers
italian   4            9800
          5            2589
spanish   4            9800
          5            2589
dtype: int64


In [ ]:
# make sure we removed all rows
print("Original image rows:", df["has_image"].sum())
print("Clean image rows:   ", clean["has_image"].sum())
print("Rejected image rows:", rejected["has_image"].sum())

assert clean["has_image"].sum() == 0, "Image rows were not fully removed from clean data"
assert rejected["has_image"].sum() == df["has_image"].sum(), "Not all image rows went to rejected"

Original image rows: 668
Clean image rows:    0
Rejected image rows: 668


In [40]:
display(clean.groupby(["language", "category"]).size())
display(clean.groupby(["language", "n_answers"]).size())
display(clean.groupby(["language", "ra_letter"]).size())

language  category    
italian   biology         2110
          chemistry       2135
          medicine        1858
          nursing         2028
          pharmacology    2145
          psychology      2113
spanish   biology         2110
          chemistry       2135
          medicine        1858
          nursing         2028
          pharmacology    2145
          psychology      2113
dtype: int64

language  n_answers
italian   4            9800
          5            2589
spanish   4            9800
          5            2589
dtype: int64

language  ra_letter
italian   A            2939
          B            3115
          C            3137
          D            2793
          E             405
spanish   A            2939
          B            3115
          C            3137
          D            2793
          E             405
dtype: int64

In [38]:
# Save
MULTILINGUAL_DIR = Path("/scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual")
out_dir = MULTILINGUAL_DIR / "data" / "processed" / "head_qa"
out_dir.mkdir(parents=True, exist_ok=True)

for col in ['aids', 'atexts']:
    clean[col] = clean[col].apply(json.dumps)
    rejected[col] = rejected[col].apply(json.dumps)

clean.to_csv(out_dir / 'head_qa_clean_all.csv', index=False)
rejected.to_csv(out_dir / 'head_qa_rejected_all.csv', index=False)

for lang, lang_name in TARGET_CONFIGS.items():
    sub = clean[clean['config'] == lang]
    sub.to_csv(out_dir / f'head_qa_clean_{lang_name}.csv', index=False)
    print(f'Saved {lang_name}: {len(sub)} rows')

Saved spanish: 12389 rows
Saved italian: 12389 rows


In [39]:
# Summary
print('=== SUMMARY ===')
print(f'Total rows (both languages): {len(df)}')
print(f'Image questions filtered:    {df["has_image"].sum()}')
print(f'Clean rows:                  {len(clean)}')
print()
print('Notes:')
print('  - Only train split exists — use 85/15 carve-out in loader')
print('  - Italian is machine-translated (Llama-3.1-8B) — check quality above')
print('  - ra maps to letter by position, not by aid value')
print('  - 4 and 5-option questions both kept (benchmark allows 2-20)')

=== SUMMARY ===
Total rows (both languages): 25502
Image questions filtered:    668
Clean rows:                  24778

Notes:
  - Only train split exists — use 85/15 carve-out in loader
  - Italian is machine-translated (Llama-3.1-8B) — check quality above
  - ra maps to letter by position, not by aid value
  - 4 and 5-option questions both kept (benchmark allows 2-20)
